# Лабораторная работа 2

Тема: сравнительное исследование методов оптимизации на гладких, мультимодальных и black-box функциях.

В работе сравниваются методы лабораторной 1 и расширения S4: расписания learning rate, Newton, BFGS, L-BFGS, Nelder-Mead, Powell и coordinate search.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

import optlib
from optlib.benchmarks import compare_methods, multistart_compare, scipy_minimize
from optlib.functions import get_objective
from optlib.plotting import plot_contours

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

print(optlib.__version__)

## Функции

Используем Розенброка, Растригина, Химмельблау, Экли, Beale, Booth, Styblinski-Tang и DesmosSurface. Для DesmosSurface точная формула содержит `round(sin(...))`, поэтому функция кусочно-гладкая и разрывная на границах полос. Для нее честный baseline — методы нулевого порядка.

In [ ]:
for name in ["rastrigin", "himmelblau", "ackley", "desmos_surface"]:
    info = optlib.BenchmarkFunctionInfo(name, 2)
    print(
        name,
        {
            key: info[key]
            for key in [
                "fixed_dimension",
                "has_gradient",
                "has_hessian",
                "is_smooth",
                "is_multimodal",
            ]
        },
    )

print("Desmos sample", optlib.DesmosSurfaceValue(np.array([1.0, 1.0])))

## Контуры и траектории 2D

На гладких функциях сравниваем L-BFGS и Adam. На DesmosSurface показываем Powell, потому что производные у этой поверхности не являются надежной информацией.

In [ ]:
plot_specs = [
    (get_objective("rosenbrock", 2), np.array([-1.2, 1.0]), "lbfgs", (-2.0, 2.0), (-1.0, 3.0)),
    (get_objective("rastrigin", 2), np.array([2.4, -1.7]), "lbfgs", (-5.12, 5.12), (-5.12, 5.12)),
    (get_objective("himmelblau"), np.array([-3.0, 3.0]), "lbfgs", (-6.0, 6.0), (-6.0, 6.0)),
    (get_objective("desmos_surface"), np.array([1.0, 1.0]), "powell", (-6.0, 6.0), (-6.0, 6.0)),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for axis, (objective, start, method, x_limits, y_limits) in zip(
    axes.ravel(), plot_specs, strict=True
):
    result = optlib.run_method(
        objective,
        start,
        method,
        max_iter=600,
        gradient_tolerance=1e-5,
        step_tolerance=1e-8,
        function_tolerance=1e-10,
        log_trajectory=True,
    )
    plot_contours(
        axis,
        objective.value,
        x_limits=x_limits,
        y_limits=y_limits,
        trajectory=result["trajectory"]["x"],
        title=f"{objective.name}: {method}",
    )
plt.tight_layout()
plt.show()

## Матрица методов и размерностей

Ниже небольшой дефолтный прогон. Для полной защиты можно увеличить список функций, размерностей и число стартов.

In [ ]:
methods = ["adam", "lbfgs", "nelder_mead", "powell"]
rows = []
for name, dimension, start in [
    ("rosenbrock", 2, np.array([-1.2, 1.0])),
    ("rastrigin", 2, np.array([0.3, -0.2])),
    ("ackley", 10, np.full(10, 0.4)),
    ("rastrigin", 50, np.full(50, 0.05)),
]:
    objective = get_objective(name, dimension)
    rows.extend(
        compare_methods(
            objective,
            start,
            methods,
            max_iter=400,
            gradient_tolerance=1e-5,
            step_tolerance=1e-8,
            function_tolerance=1e-10,
            log_trajectory=False,
        )
    )

if pd is None:
    for row in rows:
        print(row)
else:
    display(pd.DataFrame(rows))

## Multistart

Растригин мультимодален, поэтому один старт не показывает надежность метода. Считаем распределение финальных значений на фиксированном наборе стартов.

In [ ]:
rng = np.random.default_rng(42)
starts = rng.uniform(-3.0, 3.0, size=(8, 2))
rastrigin = get_objective("rastrigin", 2)
multistart_rows = multistart_compare(
    rastrigin,
    ["adam", "lbfgs", "nelder_mead", "powell"],
    starts=starts,
    max_iter=500,
    gradient_tolerance=1e-5,
    step_tolerance=1e-8,
    function_tolerance=1e-10,
    log_trajectory=False,
)

if pd is None:
    for row in multistart_rows[:8]:
        print(row)
else:
    frame = pd.DataFrame(multistart_rows)
    display(frame.groupby("method")["value"].describe())

## Сравнение со SciPy

SciPy не входит в C++ зависимости. Если установлен extra `experiments`, сравниваем с `CG`, `BFGS`, `L-BFGS-B`, `Newton-CG`, `Nelder-Mead`, `Powell`.

In [ ]:
objective = get_objective("rosenbrock", 2)
start = np.array([-1.2, 1.0])
scipy_rows = []
for method in ["CG", "BFGS", "L-BFGS-B", "Newton-CG", "Nelder-Mead", "Powell"]:
    row = scipy_minimize(objective, start, method=method, max_iter=1000)
    if row is not None:
        scipy_rows.append(row)

if not scipy_rows:
    print("SciPy is not installed; run uv sync --extra experiments --extra dev")
elif pd is None:
    for row in scipy_rows:
        print(row)
else:
    display(pd.DataFrame(scipy_rows))

## Выводы

- На гладком Розенброке квазиньютоновские методы быстрее доходят до малого градиента, чем фиксированный first-order шаг.
- На Растригине локальные методы чувствительны к старту: хороший финальный `f*` не гарантирован без multistart.
- L-BFGS масштабируется по памяти как `O(mn)` и подходит для размерностей 50 и 100.
- DesmosSurface разрывна из-за `round(sin(...))`; для нее производные около границ полос не являются честной моделью поверхности, поэтому основной baseline — zero-order методы.